#  CIC_IOT_2023 dataset — Poisson-CI Cross-Dataset Validation

Author: S. Spektor & E. Ibokete

This notebook validates the Poisson-CI detector on CIC IOT 2023 dataset.

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.poisson_ci_detector import ImprovedPoissonConcentrationML_CrossDataset
from src.features import (
    infer_protocol_ciciot,
    aggregate_time_bins_simple,
    compute_jitter_iat_features,
    compute_source_flow_concentration,
    generate_concentration_weighted_ci
)

from src.evaluation import (
    poisson_diagnostic,
    f1_at_target_fpr,
    compute_metrics
)
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings("ignore", message="Overdispersion detected in >25% of features.")

## Full Script

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

BASE_PATH = "../data"

BENIGN_FILE = os.path.join(BASE_PATH, "BenignTraffic.pcap.csv")

ATTACK_FILES = {
    "DDoS-TCP-Flood":  os.path.join(BASE_PATH, "DDoS-TCP_Flood.pcap.csv"),
    "DoS-UDP-Flood":   os.path.join(BASE_PATH, "DoS-UDP_Flood.pcap.csv"),
    "Recon-PortScan":  os.path.join(BASE_PATH, "Recon-PortScan.pcap.csv"),
    "BruteForce-SSH":  os.path.join(BASE_PATH, "DictionaryBruteForce.pcap.csv"),
}

# ------------------------------------------------------------
# EXPERIMENT SETTINGS
# ------------------------------------------------------------
BIN_WIDTH = "5ms"
ALPHA = 0.05
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------
# TRACKING CONTAINERS
# ------------------------------------------------------------
kappa_results = {}

## LOAD AND INSPECT BENIGN DATA

In [ ]:
# ============================================================
# LOAD BENIGN DATA
# ============================================================

df_benign = pd.read_csv(BENIGN_FILE, low_memory=False)

required_benign_cols = ["IAT"]
missing_cols = [c for c in required_benign_cols if c not in df_benign.columns]

if missing_cols:
    raise ValueError(f"Missing required benign columns: {missing_cols}")

## RECONSTRUCT TIMESTAMPS AND BIN

In [ ]:
# ============================================================
# TIMESTAMP RECONSTRUCTION AND BINNING
# ============================================================

iat_col = "IAT"

if iat_col not in df_benign.columns:
    raise ValueError(f"Required column '{iat_col}' not found.")

df_benign[iat_col] = pd.to_numeric(df_benign[iat_col], errors="coerce").fillna(0)

df_benign["time_elapsed_s"] = df_benign[iat_col].cumsum()

base_time = pd.Timestamp("2025-01-01")

df_benign["Timestamp_parsed"] = base_time + pd.to_timedelta(
    df_benign["time_elapsed_s"], unit="s"
)

df_benign["time_bin"] = df_benign["Timestamp_parsed"].dt.floor(BIN_WIDTH)

# Minimal sanity check (no prints)
assert df_benign["time_bin"].notna().all()

## INFER PROTOCOL AND AGGREGATE

In [ ]:
# ============================================================
# PROTOCOL INFERENCE AND AGGREGATION
# ============================================================

df_benign = infer_protocol_ciciot(df_benign)

agg = aggregate_time_bins_simple(df_benign)

required_agg_cols = ["time_bin", "event_count", "TCP", "UDP", "ICMP"]
missing_cols = [c for c in required_agg_cols if c not in agg.columns]

if missing_cols:
    raise ValueError(f"Aggregation missing required columns: {missing_cols}")

## VMR DIAGNOSTICS

In [ ]:
# ============================================================
# VMR DIAGNOSTICS
# ============================================================

vmr_results = poisson_diagnostic(
    agg,
    label="Benign (5 ms bins)",
    verbose=False
)

## FEATURE ENGINEERING

In [ ]:
# ============================================================
# FEATURE ENGINEERING AND TEMPORAL SPLIT
# ============================================================

for col in ["event_count", "TCP", "UDP", "ICMP"]:
    agg[f"log_{col}"] = np.log1p(agg[col])

agg["target"] = agg["log_event_count"]

feature_cols = [
    "TCP", "UDP", "ICMP",
    "log_TCP", "log_UDP", "log_ICMP"
]

X = agg[feature_cols].values.astype(float)
y = agg["target"].values.astype(float)

n = len(X)
n_train = int(0.6 * n)
n_val = int(0.2 * n)

X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:n_train + n_val], y[n_train:n_train + n_val]
X_test, y_test = X[n_train + n_val:], y[n_train + n_val:]

assert X_train.shape[1] == len(feature_cols)
assert len(X_train) > 0 and len(X_val) > 0 and len(X_test) > 0

## POISSON-CI DETECTOR (dont run yet)

In [ ]:
# ============================================================
# TRAIN POISSON-CI DETECTOR
# ============================================================
detector = ImprovedPoissonConcentrationML_CrossDataset(
    q=2,
    confidence_level=0.95,
    lambda_mode="input_based",
    empirical_calibration=True,
    verbose=False
)

detector.fit(X_train, y_train)

detector.kappa = float(detector.calibration_factor)

if hasattr(detector, "raw_kappa_"):
    if np.isnan(detector.raw_kappa_) or np.isinf(detector.raw_kappa_):
        raise ValueError("Raw kappa is unstable: NaN or Inf")

    kappa_results[BIN_WIDTH] = detector.raw_kappa_
else:
    raise AttributeError("Detector did not expose raw_kappa_.")

## EVALUATE ON BENIGN TEST SET (FPR)

In [ ]:
# ============================================================
# BENIGN TEST SET — CALIBRATION
# ============================================================

preds_test, ci_test = detector.predict_with_confidence(X_test)

assert ci_test.shape == (len(X_test), 2)
assert np.all(ci_test[:, 1] >= ci_test[:, 0])

lower_test = ci_test[:, 0]
upper_test = ci_test[:, 1]

violations = (y_test < lower_test) | (y_test > upper_test)

fpr = np.mean(violations)
coverage = 1 - fpr

ci_width = upper_test - lower_test

print(f"[RESULT] Benign calibration:")
print(f"  FPR       : {fpr:.6f}")
print(f"  Coverage  : {coverage:.6f}")
print(f"  Mean CI   : {ci_width.mean():.4f}")
print(f"  Target α  : {ALPHA:.3f}")

[RESULT] Benign calibration:
  FPR       : 0.011442
  Coverage  : 0.988558
  Mean CI   : 0.3766
  Target α  : 0.050


## EVALUATE ON ATTACK DATA

In [ ]:
# ============================================================
# ATTACK EVALUATION
# ============================================================
iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=RANDOM_STATE)
iso.fit(X_train)

ocsvm = OneClassSVM(kernel="rbf", nu=0.05)
subsample_idx = np.random.choice(len(X_train), min(3000, len(X_train)), replace=False)
ocsvm.fit(X_train[subsample_idx])

all_attack_results = []
global_X_attack, global_y_attack, global_event_count_attack = [], [], []

for attack_name, attack_file in ATTACK_FILES.items():
    if not os.path.exists(attack_file):
        raise FileNotFoundError(f"Attack file not found: {attack_file}")

    df_atk = pd.read_csv(attack_file, low_memory=False)

    if iat_col in df_atk.columns:
        df_atk[iat_col] = pd.to_numeric(df_atk[iat_col], errors="coerce").fillna(0)
        df_atk["time_elapsed_s"] = df_atk[iat_col].cumsum()
        df_atk["Timestamp_parsed"] = pd.Timestamp("2025-06-01") + pd.to_timedelta(
            df_atk["time_elapsed_s"], unit="s"
        )
        df_atk["time_bin"] = df_atk["Timestamp_parsed"].dt.floor(BIN_WIDTH)
    else:
        df_atk["time_bin"] = pd.date_range("2025-06-01", periods=len(df_atk), freq=BIN_WIDTH)

    df_atk = infer_protocol_ciciot(df_atk)

    agg_atk = df_atk.groupby("time_bin").agg(
        event_count=("Protocol", "size"),
        TCP=("Protocol", lambda x: (x == "TCP").sum()),
        UDP=("Protocol", lambda x: (x == "UDP").sum()),
        ICMP=("Protocol", lambda x: (x == "ICMP").sum()),
        OTHER=("Protocol", lambda x: (x == "OTHER").sum()),
    ).reset_index().fillna(0).sort_values("time_bin").reset_index(drop=True)

    for col in ["event_count", "TCP", "UDP", "ICMP"]:
        agg_atk[f"log_{col}"] = np.log1p(agg_atk[col])

    agg_atk["target"] = agg_atk["log_event_count"]

    X_attack = agg_atk[feature_cols].values.astype(float)
    y_attack = agg_atk["target"].values.astype(float)
    event_count_attack = agg_atk["event_count"].values.astype(float)

    global_X_attack.append(X_attack)
    global_y_attack.append(y_attack)
    global_event_count_attack.append(event_count_attack)

    combined_features = np.vstack([X_test, X_attack])
    combined_targets = np.concatenate([y_test, y_attack])
    combined_labels_local = np.concatenate([
        np.zeros(len(X_test), dtype=int),
        np.ones(len(X_attack), dtype=int)
    ])

    _, ci_c = detector.predict_with_confidence(combined_features)
    pci_flags = (combined_targets < ci_c[:, 0]) | (combined_targets > ci_c[:, 1])

    iso_flags = iso.predict(combined_features) == -1
    ocsvm_flags = ocsvm.predict(combined_features) == -1

    for method_name, flags in [
        ("Poisson-CI", pci_flags),
        ("IsoForest", iso_flags),
        ("OC-SVM", ocsvm_flags)
    ]:
        m = compute_metrics(flags, combined_labels_local)
        m["Method"] = method_name
        m["Attack"] = attack_name
        m["N_attack_bins"] = len(X_attack)
        all_attack_results.append(m)

if not global_X_attack:
    raise ValueError("No attack files were loaded.")

X_attack_all = np.vstack(global_X_attack)
y_attack_all = np.concatenate(global_y_attack)
event_count_attack_all = np.concatenate(global_event_count_attack)

X_combined = np.vstack([X_test, X_attack_all])
y_combined = np.concatenate([y_test, y_attack_all])

combined_labels = np.concatenate([
    np.zeros(len(X_test), dtype=int),
    np.ones(len(X_attack_all), dtype=int)
])

event_count_train = np.expm1(y_train)
event_count_test = np.expm1(y_test)
event_count_combined = np.concatenate([event_count_test, event_count_attack_all])

if hasattr(detector, "scaler_"):
    X_train_scaled = detector.scaler_.transform(X_train)
    X_combined_scaled = detector.scaler_.transform(X_combined)
else:
    X_train_scaled = X_train
    X_combined_scaled = X_combined

assert X_combined.shape[0] == y_combined.shape[0] == combined_labels.shape[0]
assert X_train_scaled.shape[1] == X_combined_scaled.shape[1]

## ROC Curves and F1 at Matched FPR

In [ ]:
# ============================================================
# GLOBAL ROC AND MATCHED-FPR EVALUATION
# ============================================================

TARGET_FPR = 0.01

roc_results = {}

preds_poisson, _ = detector.predict_with_confidence(X_combined)

lam = np.clip(np.expm1(preds_poisson), 1e-6, None)
y_lambda = np.expm1(y_combined)

scores_poisson = (
    y_lambda * np.log((y_lambda + 1e-6) / lam) - (y_lambda - lam)
)

roc_results["Poisson-CI"] = f1_at_target_fpr(
    combined_labels,
    scores_poisson,
    TARGET_FPR
)

iso_roc = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=RANDOM_STATE
)
iso_roc.fit(X_train_scaled)

scores_iso = -iso_roc.decision_function(X_combined_scaled)

roc_results["IsoForest"] = f1_at_target_fpr(
    combined_labels,
    scores_iso,
    TARGET_FPR
)

n_sub = min(3000, len(X_train_scaled))
idx_sub = np.random.choice(len(X_train_scaled), n_sub, replace=False)

ocsvm_roc = OneClassSVM(kernel="rbf", nu=0.05)
ocsvm_roc.fit(X_train_scaled[idx_sub])

scores_ocsvm = -ocsvm_roc.decision_function(X_combined_scaled)

roc_results["OC-SVM"] = f1_at_target_fpr(
    combined_labels,
    scores_ocsvm,
    TARGET_FPR
)

train_mean = np.mean(event_count_train)
train_std = np.std(event_count_train) + 1e-10

scores_event_3sig = np.abs(event_count_combined - train_mean) / train_std

roc_results["EventCount-3sigma"] = f1_at_target_fpr(
    combined_labels,
    scores_event_3sig,
    TARGET_FPR
)

z_scores = np.zeros_like(X_combined_scaled)

for col in range(X_combined_scaled.shape[1]):
    col_mean = np.mean(X_train_scaled[:, col])
    col_std = np.std(X_train_scaled[:, col]) + 1e-10
    z_scores[:, col] = np.abs(X_combined_scaled[:, col] - col_mean) / col_std

scores_layered = np.max(z_scores, axis=1)

roc_results["Layered-3sigma"] = f1_at_target_fpr(
    combined_labels,
    scores_layered,
    TARGET_FPR
)

df_roc_results = pd.DataFrame.from_dict(roc_results, orient="index").reset_index()
df_roc_results = df_roc_results.rename(columns={"index": "Method"})

#display(df_roc_results)

## SUMMARY TABLE

In [ ]:
# ============================================================
# FINAL SUMMARY TABLES
# ============================================================

if not all_attack_results:
    raise ValueError("No attack results available. Step 8 may have skipped all attacks.")

df_results = pd.DataFrame(all_attack_results)

# ------------------------------------------------------------
# A. CALIBRATION
# ------------------------------------------------------------
print("\n=== Calibration: Benign Test Set ===")
print(f" Target FPR (alpha): {ALPHA:.3f}")
print(f" Observed FPR      : {fpr:.6f}")
print(f" Coverage          : {coverage:.6f}")
print(f" Mean CI Width     : {np.mean(ci_test[:, 1] - ci_test[:, 0]):.4f}")

# ------------------------------------------------------------
# B. PER-ATTACK POISSON-CI
# ------------------------------------------------------------
print("\n=== Per-Attack Poisson-CI Results ===")
pci_only = df_results[df_results["Method"] == "Poisson-CI"]

for _, row in pci_only.iterrows():
    print(
        f"  {row['Attack']:25s}: "
        f"Prec={row['Prec']:.4f} "
        f"Rec={row['Rec']:.4f} "
        f"F1={row['F1']:.4f} "
        f"FPR={row['FPR']:.6f} "
        f"TP={int(row['TP'])} "
        f"FN={int(row['FN'])}"
    )

# ------------------------------------------------------------
# C. PER-ATTACK MODEL COMPARISON
# ------------------------------------------------------------
print("\n=== Per-Attack Model Comparison ===")

for attack in df_results["Attack"].unique():
    sub = df_results[df_results["Attack"] == attack]

    print(f"\n  --- {attack} ---")

    for method in ["Poisson-CI", "IsoForest", "OC-SVM"]:
        method_rows = sub[sub["Method"] == method]

        if method_rows.empty:
            continue

        row = method_rows.iloc[0]

        print(
            f"    {method:12s}: "
            f"Prec={row['Prec']:.4f} "
            f"Rec={row['Rec']:.4f} "
            f"F1={row['F1']:.4f} "
            f"FPR={row['FPR']:.6f}"
        )

# ------------------------------------------------------------
# D. WEIGHTED AVERAGE
# ------------------------------------------------------------
print("\n=== Weighted Average Across All Attacks ===")

for method in ["Poisson-CI", "IsoForest", "OC-SVM"]:
    mdf = df_results[df_results["Method"] == method]

    if mdf.empty:
        continue

    weights = mdf["N_attack_bins"].values

    if weights.sum() == 0:
        continue

    w_prec = np.average(mdf["Prec"], weights=weights)
    w_rec = np.average(mdf["Rec"], weights=weights)
    w_f1 = 2 * w_prec * w_rec / (w_prec + w_rec + 1e-10)
    w_fpr = np.average(mdf["FPR"], weights=weights)

    print(
        f"  {method:12s}: "
        f"Prec={w_prec:.4f} "
        f"Rec={w_rec:.4f} "
        f"F1={w_f1:.4f} "
        f"FPR={w_fpr:.6f}"
    )

# ------------------------------------------------------------
# F. VMR TABLE
# ------------------------------------------------------------
print("\n=== VMR Table ===")
vmr_results = poisson_diagnostic(
    agg,
    label="Benign (5 ms bins)",
    verbose=True
)

# ------------------------------------------------------------
# G. KAPPA
# ------------------------------------------------------------
print("\n=== Kappa ===")

if hasattr(detector, "raw_kappa_"):
    print(f"  raw kappa   = {detector.raw_kappa_:.6f}")
else:
    print("  raw kappa not available")

if hasattr(detector, "kappa"):
    print(f"  final kappa = {detector.kappa:.3f}")
else:
    print("  final kappa not available")


# ------------------------------------------------------------
# E. GLOBAL ROC / MATCHED-FPR RESULTS
# ------------------------------------------------------------
print("\n=== Global ROC / Matched-FPR Results ===")
#print(df_roc_results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
display(df_roc_results)


=== Calibration: Benign Test Set ===
 Target FPR (alpha): 0.050
 Observed FPR      : 0.011442
 Coverage          : 0.988558
 Mean CI Width     : 0.3766

=== Per-Attack Poisson-CI Results ===
  DDoS-TCP-Flood           : Prec=0.7946 Rec=0.7968 F1=0.7957 FPR=0.012571 TP=2627 FN=670
  DoS-UDP-Flood            : Prec=0.7957 Rec=0.8091 F1=0.8024 FPR=0.012571 TP=2645 FN=624
  Recon-PortScan           : Prec=0.5658 Rec=0.0162 F1=0.0315 FPR=0.011609 TP=817 FN=49542
  BruteForce-SSH           : Prec=0.2246 Rec=0.0159 F1=0.0296 FPR=0.011442 TP=179 FN=11113

=== Per-Attack Model Comparison ===

  --- DDoS-TCP-Flood ---
    Poisson-CI  : Prec=0.7946 Rec=0.7968 F1=0.7957 FPR=0.012571
    IsoForest   : Prec=0.5974 Rec=0.8201 F1=0.6913 FPR=0.033733
    OC-SVM      : Prec=0.1564 Rec=0.8720 F1=0.2653 FPR=0.287010

  --- DoS-UDP-Flood ---
    Poisson-CI  : Prec=0.7957 Rec=0.8091 F1=0.8024 FPR=0.012571
    IsoForest   : Prec=0.6369 Rec=0.9777 F1=0.7713 FPR=0.033733
    OC-SVM      : Prec=0.1721 Rec=0.98

,Method,AUC,F1@FPR=0.01,Recall@FPR=0.01,ActualFPR
0,Poisson-CI,0.536147,0.165815,0.091297,0.012497
1,IsoForest,0.540599,0.131261,0.071067,0.014867
2,OC-SVM,0.577750,0.204554,0.114839,0.010090
3,EventCount-3sigma,0.531792,0.226896,0.129997,0.020051
4,Layered-3sigma,0.542145,0.212536,0.120087,0.012571


### KAPPA TABLE

In [ ]:
# ============================================================
# RAW KAPPA TABLE
# ============================================================

df_kappa = pd.DataFrame([
    {"Bin width": bw, "Raw kappa": val}
    for bw, val in kappa_results.items()
])

print("\n=== Raw Kappa Table ===")
print(df_kappa.to_string(index=False, float_format=lambda x: f"{x:.8f}"))


=== Raw Kappa Table ===
Bin width  Raw kappa
      5ms 0.14049228
